# Two text corpora, same labels (HF + PMH)

**Story:** Classify/support tickets from **Product A** vs **Product B** wording. Labels unchanged.

Toy encoder runs without GPU/download. Swap in your `AutoModel` + real text lists.

## Parameters cheat sheet (HF / LLM)

| Knob | Typical |
|------|---------|
| `rank` | 32 |
| `PMHConfig` | `.finetune_llm()` or `.balanced()` |
| `nuisance` | `domain_shift` (two corpora) or `style` (format pairs) |

```python
from pmh import PMHConfig
PMHConfig.finetune_llm()  # long warmup for transformers
```

[PARAMETERS_CHEATSHEET.md](https://github.com/vishalstark512/matching-pmh/blob/main/docs/PARAMETERS_CHEATSHEET.md) · `pmh-train doctor --stack hf`

In [ ]:
!pip install -q "matching-pmh[hf]"

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from pmh import check_applicability


class ToyTokenizer:
    pad_token = "<pad>"
    eos_token = "<pad>"

    def __call__(self, texts, return_tensors="pt", padding=True, truncation=True, max_length=32):
        import hashlib
        rows = []
        for t in texts:
            h = int(hashlib.md5(t.encode()).hexdigest()[:8], 16)
            torch.manual_seed(h % (2**31))
            rows.append(torch.randn(32))
        ids = torch.stack(rows)
        return {"input_ids": ids, "attention_mask": torch.ones(ids.shape[0], ids.shape[1])}


class HashEncoder(nn.Module):
    def __init__(self, d=32, n_classes=3):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(d, d), nn.ReLU())
        self.lm_head = nn.Linear(d, n_classes)

    def forward(self, input_ids=None, labels=None, output_hidden_states=False, **kw):
        h = self.body(input_ids)
        logits = self.lm_head(h)
        # [B, T, d] for HF hidden_states convention (T=1 here)
        return type("O", (), {"logits": logits, "hidden_states": (h.unsqueeze(1),)})


class TextDS(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        tok = ToyTokenizer()
        ids = tok([self.texts[i]])["input_ids"][0]
        return {"input_ids": ids, "labels": torch.tensor(self.labels[i])}


texts_a = [f"product A issue type {i % 3}" for i in range(80)]
texts_b = [f"PRODUCT-B ticket #{i} category {i % 3}" for i in range(80)]
labels = [i % 3 for i in range(80)]
train_loader = DataLoader(TextDS(texts_a, labels), batch_size=16, shuffle=True)

print(check_applicability(stack="hf", n_source=len(texts_a), n_target=len(texts_b)).summary())

In [ ]:
from pmh.hf_trainer import HFPMHTrainer
from pmh.onboarding import preflight_plain_english

model = HashEncoder()
tokenizer = ToyTokenizer()

trainer = HFPMHTrainer(model, tokenizer, nuisance="domain_shift", rank=4)
trainer.estimate_text_domains(texts_a, texts_b)
pf = trainer.artifact_.preflight
print(f"Preflight: {pf} - {preflight_plain_english(pf)}")
print("Phase B training: see Walkthrough 7 or robust_fit_text_domains with your HF Trainer.")

## Production

- Replace toy model with `AutoModelForSequenceClassification` / causal LM + your tokenizer.
- Use `pmh.integrations.hf_trainer.get_pmh_trainer()` for full HF `Trainer`.
- [Golden path G3](https://github.com/vishalstark512/matching-pmh/blob/main/docs/GOLDEN_PATHS.md)